# 04 — Budget Optimization

Allocates a fixed subsidy budget across tier x voucher-config options via
a greedy fractional knapsack (optimal for the LP relaxation), benchmarks
against the PDF's 3 named strategies, and stress-tests the recommendation
against the 3 unvalidated business assumptions. See `budget_allocator.py`
for full docstrings and the framing note on why this is a hard-budget-cap
problem, not a "spend until marginal ROI hits zero" problem.

In [1]:
import sys

sys.path.insert(0, "..")

import pandas as pd

from data_generation import VoucherDGP
from budget_allocator import (
    build_tier_condition_table,
    greedy_fractional_allocation,
    evaluate_fixed_strategy,
    sensitivity_analysis,
    BUDGET_TWD,
    BUDGET_USD,
    TOTAL_ADDRESSABLE_USERS,
)
from visualization import plot_tier_lever_heatmap, plot_budget_efficiency_frontier
from experiment_analysis import run_named_comparisons

calibration = pd.read_csv("../data/raw_benchmarks/case_summary_tables.csv")
experiment_log = pd.read_csv("../data/processed/experiment_log.csv")
dgp = VoucherDGP(calibration_path="../data/raw_benchmarks/case_summary_tables.csv")

print(f"Budget: NT${BUDGET_TWD:,.0f} = ${BUDGET_USD:,.2f} USD")
print(f"Addressable population: {TOTAL_ADDRESSABLE_USERS:,} users")

Budget: NT$1,000,000 = $31,746.03 USD
Addressable population: 500,000 users


## Economics table: expected profit, cost, and ROI ratio per (tier, condition)

In [2]:
econ_table = build_tier_condition_table(dgp, calibration)
econ_table[
    ["tier", "condition", "profit_per_user", "cost_per_user", "incremental_profit_per_user", "roi_ratio"]
].sort_values(["tier", "roi_ratio"], ascending=[True, False])

,tier,condition,profit_per_user,cost_per_user,incremental_profit_per_user,roi_ratio
2,0-29,s0_m249_c3,0.147000,7.380000,0.000000,0.000000
3,0-29,s0_m299_c3,0.147000,7.470000,0.000000,0.000000
4,0-29,s9_m249_c3,0.146488,5.151079,-0.000512,-0.000099
6,0-29,s19_m199_c3,0.145868,2.656500,-0.001132,-0.000426
1,0-29,s0_m199_c1,0.145888,2.400000,-0.001112,-0.000463
5,0-29,s19_m199_c1,0.144797,0.874500,-0.002203,-0.002519
0,0-29,no_voucher,0.147000,0.000000,0.000000,-inf
40,100,s19_m199_c1,0.028642,0.351389,0.000642,0.001826
41,100,s19_m199_c3,0.027747,1.087167,-0.000253,-0.000233
39,100,s9_m249_c3,0.027344,2.146974,-0.000656,-0.000305


## Our allocation: greedy fractional knapsack

In [3]:
alloc_df, summary = greedy_fractional_allocation(econ_table)
alloc_df

,tier,condition,n_allocated,cost,incremental_profit,roi_ratio
0,70-79,s19_m199_c1,28417,31744.94,268.06,0.01
1,100,s19_m199_c1,3,1.05,0.00,0.00


In [4]:
print(f"Total cost: ${summary['total_cost']:,.2f} ({summary['budget_utilization_pct']:.1f}% of budget)")
print(f"Total incremental profit: ${summary['total_incremental_profit']:,.2f}")
print(f"Blended ROI: {summary['blended_roi_pct']:.1f}%")

Total cost: $31,745.99 (100.0% of budget)
Total incremental profit: $268.06
Blended ROI: 0.8%


## Benchmark: PDF's 3 named strategies at the same budget

In [5]:
scenario_1 = evaluate_fixed_strategy(econ_table, ["90-99"], "s0_m249_c3", BUDGET_USD)
scenario_2 = evaluate_fixed_strategy(econ_table, ["30-69"], "s9_m249_c3", BUDGET_USD)
scenario_3_high = evaluate_fixed_strategy(econ_table, ["90-99"], "s0_m249_c3", BUDGET_USD * 0.6)
scenario_3_mod = evaluate_fixed_strategy(econ_table, ["30-69"], "s9_m249_c3", BUDGET_USD * 0.4)
scenario_3 = {
    "total_cost": scenario_3_high["total_cost"] + scenario_3_mod["total_cost"],
    "total_incremental_profit": scenario_3_high["total_incremental_profit"]
    + scenario_3_mod["total_incremental_profit"],
    "users_reached": scenario_3_high["users_reached"] + scenario_3_mod["users_reached"],
}
scenario_3["blended_roi_pct"] = (
    round(scenario_3["total_incremental_profit"] / scenario_3["total_cost"] * 100, 1)
    if scenario_3["total_cost"] > 0
    else 0.0
)

benchmark_table = pd.DataFrame(
    [
        {"strategy": "PDF Scenario 1 (order-volume max)", **scenario_1},
        {"strategy": "PDF Scenario 2 (profit-per-user max)", **scenario_2},
        {"strategy": "PDF Scenario 3 (60/40 blended)", **scenario_3},
        {"strategy": "This project's ROI-ranked allocation", **summary},
    ]
)
benchmark_table

,strategy,total_cost,total_incremental_profit,users_reached,budget_utilization_pct,blended_roi_pct,budget
0,PDF Scenario 1 (order-volume max),31740.57,6.24,3121,100.000000,0.00000,NaN
1,PDF Scenario 2 (profit-per-user max),31741.00,-44.35,5077,100.000000,-0.10000,NaN
2,PDF Scenario 3 (60/40 blended),31735.89,-14.00,3903,NaN,-0.00000,NaN
3,This project's ROI-ranked allocation,31745.99,268.06,28420,99.999869,0.84439,31746.031746


## Sensitivity analysis: does the recommendation hold under different assumptions?

In [6]:
sensitivity = sensitivity_analysis(calibration, calibration_path="../data/raw_benchmarks/case_summary_tables.csv")
sensitivity.to_csv("../outputs/sensitivity_analysis.csv", index=False)
sensitivity

,shipping_base_cost,total_addressable_users,twd_per_usd,budget_usd,top_allocated_segment,total_incremental_profit,blended_roi_pct,users_reached
0,20.0,250000,28.0,35714.29,70-79/s19_m199_c1,575.06,1.61,121071
1,20.0,250000,31.5,31746.03,70-79/s19_m199_c1,568.07,1.79,120435
2,20.0,250000,35.0,28571.43,70-79/s19_m199_c1,562.47,1.97,119927
3,20.0,500000,28.0,35714.29,70-79/s19_m199_c1,1087.19,3.04,236418
4,20.0,500000,31.5,31746.03,70-79/s19_m199_c1,1080.20,3.40,235782
5,20.0,500000,35.0,28571.43,70-79/s19_m199_c1,1074.60,3.76,235273
6,20.0,1000000,28.0,35714.29,70-79/s19_m199_c1,2111.43,5.91,467114
7,20.0,1000000,31.5,31746.03,70-79/s19_m199_c1,2050.30,6.46,374516
8,20.0,1000000,35.0,28571.43,70-79/s19_m199_c1,1986.50,6.95,275136
9,30.0,250000,28.0,35714.29,70-79/s19_m199_c1,266.33,0.75,31479


In [7]:
n_unique = sensitivity["top_allocated_segment"].nunique()
print(
    f"Top-allocated segment is identical across all {len(sensitivity)} assumption "
    f"combinations tested ({n_unique} unique choice)"
    if n_unique == 1
    else f"Top-allocated segment varies across {n_unique} different choices — recommendation is assumption-sensitive."
)

Top-allocated segment is identical across all 27 assumption combinations tested (1 unique choice)


## Visualizations

In [8]:
comparisons = run_named_comparisons(experiment_log)
plot_tier_lever_heatmap(comparisons, "../outputs/figures/tier_lever_heatmap.png")
plot_budget_efficiency_frontier(
    dgp, calibration, "../outputs/figures/budget_efficiency_frontier.png", max_budget=BUDGET_USD * 5
)
print("Figures saved to outputs/figures/")

Figures saved to outputs/figures/
